In [5]:
import sys
sys.path.insert(0, "/home/icb/zaikang.lin/projects/cellflow_lung_organoids/CellFlow_pheno/src")


import warnings
from pandas.errors import SettingWithCopyWarning

warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", FutureWarning)
warnings.simplefilter("ignore", SettingWithCopyWarning)

import numpy as np
import pandas as pd
import seaborn as sns
import jax
import functools
import matplotlib.pyplot as plt
import anndata as ad
import scanpy as sc
import flax.linen as nn
import optax
import cellflow
from cellflow.model import CellFlow
import cellflow.preprocessing as cfpp
from cellflow.utils import match_linear


In [6]:
print(cellflow.__file__)

/home/icb/zaikang.lin/projects/cellflow_lung_organoids/CellFlow_pheno/src/cellflow/__init__.py


In [7]:
#adata = sc.read_h5ad('/home/icb/zaikang.lin/projects/cellflow_lung_organoids/toydata/toy_adata.h5ad')

In [8]:
"""""
control_organoids = ['Round_1_13_PC_filtered', 
                     'Round_1_PC_a_filtered', 
                     'Round_1_PC_matrigel_filtered',
                     'Round_1_PCs1_filtered', 
                     'Round_2_BD2_filtered', 
                     'Round_2_BD_a_filtered', 
                     'Round_2_BD_matrigel_filtered', 
                     'Round_2_BDs_3_and_BDs_5_filtered', 
                     'Round_2_PC_PDMS_filtered',
                     'Round_2_PCs_2_and_PCs_3_filtered',
                     'Round_2_PCs_4_filtered',
                     'Round_2_PCs_5_filtered',
                     'Round_3_13_filtered',
                     'Round_3_13s_PC_filtered',
                     'Round_3_BDs_1_filtered', 
                     'Round_3_BDs_4_filtered', 
                     'Round_3_BDs_6_filtered'
                    ]

"""""

'""\ncontrol_organoids = [\'Round_1_13_PC_filtered\', \n                     \'Round_1_PC_a_filtered\', \n                     \'Round_1_PC_matrigel_filtered\',\n                     \'Round_1_PCs1_filtered\', \n                     \'Round_2_BD2_filtered\', \n                     \'Round_2_BD_a_filtered\', \n                     \'Round_2_BD_matrigel_filtered\', \n                     \'Round_2_BDs_3_and_BDs_5_filtered\', \n                     \'Round_2_PC_PDMS_filtered\',\n                     \'Round_2_PCs_2_and_PCs_3_filtered\',\n                     \'Round_2_PCs_4_filtered\',\n                     \'Round_2_PCs_5_filtered\',\n                     \'Round_3_13_filtered\',\n                     \'Round_3_13s_PC_filtered\',\n                     \'Round_3_BDs_1_filtered\', \n                     \'Round_3_BDs_4_filtered\', \n                     \'Round_3_BDs_6_filtered\'\n                    ]\n\n'

Let's investigate the data:

In [9]:
#adata.obs.head()

We first create a column which saved the experimental condition, i.e. the combination of donor and treatment.
Moreover, we require a boolean column which indicates if a cell is in control or perturbed state.

In [10]:
#list(adata.obs)

In [11]:
"""""
adata.obs["condition"] = adata.obs.apply(lambda x: x["sample"], axis=1)
adata.obs["is_control"] = adata.obs.apply(lambda x: True if x["sample"] in control_organoids else False, axis=1)
"""""

'""\nadata.obs["condition"] = adata.obs.apply(lambda x: x["sample"], axis=1)\nadata.obs["is_control"] = adata.obs.apply(lambda x: True if x["sample"] in control_organoids else False, axis=1)\n'

We then normalize the data to a constant library size.

Similarly to the use case in the CellFlow manuscript, we aim to predict the response of donors to the IL-15 treatment. As we found performance to increase significantly as soon as the cytokine has been observed for one donor, we include the cytokine treatment for Donor 8, and predict the responses of the remaining donors.

Therefore, we split our training data into training and test data. We note that the test data has to include the control populations ("PBS") of all donors whose response we want to predict. For this notebook, we restrict to predicting the response of donor 1.

In [12]:
"""""
adata_train = adata[(adata.obs["condition"]!="Round_1_31_filtered")].copy()
adata_test = adata[adata.obs["condition"]=="Round_2_75s_filtered"].copy()
adata_train.n_obs, adata_test.n_obs
"""""

'""\nadata_train = adata[(adata.obs["condition"]!="Round_1_31_filtered")].copy()\nadata_test = adata[adata.obs["condition"]=="Round_2_75s_filtered"].copy()\nadata_train.n_obs, adata_test.n_obs\n'

In [13]:
#adata_train.n_obs, adata_test.n_obs

We now compute PCA on the training data, and then project the test data onto it. CellFlow implements these functions with GPU acceleration, which we can leverage using `method="rapids"`.

In [14]:
"""""
cfpp.centered_pca(adata_train, n_comps=100, keep_centered_data=False)
cfpp.project_pca(query_adata=adata_test, ref_adata=adata_train)
"""""

'""\ncfpp.centered_pca(adata_train, n_comps=100, keep_centered_data=False)\ncfpp.project_pca(query_adata=adata_test, ref_adata=adata_train)\n'

In [15]:
#adata_train.write_h5ad('/home/icb/zaikang.lin/projects/cellflow_lung_organoids/toydata/adata_toy_train.h5ad')

## Setting up the CellFlow model

We are now ready to setup the {class}`~cellflow.model.CellFlow` model.

Therefore, we first choose the flow matching solver. We select the solver `"otfm"`, which deterministically maps a cell to its perturbed equivalent. If we wanted to incorporate stochasticity on single-cell level, we would select `"genot"`.

In [16]:
adata_train = sc.read_h5ad('/home/icb/zaikang.lin/projects/cellflow_lung_organoids/toydata/adata_toy_train.h5ad')

In [17]:
control_organoids = ['Round_1_13_PC_filtered', 
                     'Round_1_PC_a_filtered', 
                     'Round_1_PC_matrigel_filtered',
                     'Round_1_PCs1_filtered', 
                     'Round_2_BD2_filtered', 
                     'Round_2_BD_a_filtered', 
                     'Round_2_BD_matrigel_filtered', 
                     'Round_2_BDs_3_and_BDs_5_filtered', 
                     'Round_2_PC_PDMS_filtered',
                     'Round_2_PCs_2_and_PCs_3_filtered',
                     'Round_2_PCs_4_filtered',
                     'Round_2_PCs_5_filtered',
                     'Round_3_13_filtered',
                     'Round_3_13s_PC_filtered',
                     'Round_3_BDs_1_filtered', 
                     'Round_3_BDs_4_filtered', 
                     'Round_3_BDs_6_filtered'
                    ]

donor_key = "donor_id_transfer"

# Example: one scalar phenotype per donor (replace with your real phenotype!)
donors = adata_train.obs[donor_key].astype(str).unique()
donor_to_vec =  {d: np.array([i], dtype=np.float32) for i, d in enumerate(sorted(donors))}

adata_train.uns["donor_id_transfer"] = donor_to_vec

adata_train.obs["condition"] = adata_train.obs.apply(lambda x: x["sample"], axis=1)
adata_train.obs["is_control"] = adata_train.obs.apply(lambda x: True if x["sample"] in control_organoids else False, axis=1)

import cellflow.preprocessing as cfpp

# Run on full adata so test conditions are also encoded
cfpp.encode_onehot(
    adata_train,
    covariate_keys="condition",          # column in adata.obs
    uns_key_added="drug_treatment_onehot",
    exclude_values=control_organoids,    # don't encode control samples
)

In [18]:
cfpp.encode_onehot(
    adata_train,
    covariate_keys="donor_id_transfer",          # column in adata.obs
    uns_key_added="donor_id_transfer_onehot",
    exclude_values=control_organoids,    # don't encode control samples
)

In [19]:
len(adata_train.uns['donor_id_transfer_onehot'])

236

In [20]:
cf = CellFlow(adata_train, solver="otfm")

## Preparing {class}`~cellflow.model.CellFlow`'s data handling with {meth}`~cellflow.model.CellFlow.prepare_data`

We now prepare the data. Therefore, we have to choose the sample representation, i.e. the space the (measure and generated) cells live in. We use {attr}`obsm['X_pca'] <anndata.AnnData.obsm>` as computed above. Moreover, we set the control key to {attr}`obs['is_control'] <anndata.AnnData.obs>`, as defined above.

The `perturbation_covariates` indicates the external intervention, i.e. the cytokine treatment. We define a key (of arbitrary name) `"cytokine_treatment"` for this, and have the values be tuples with the perturbation and potential perturbation covariates. As we don't have a perturbation covariate (e.g. always the same dose), we only have one tuple, and as we don't observe combinations of treatments, the tuple has length 1. We use ESM2 embeddings for representing the cytokines, which we have precomputed already for the purpose of this notebook, saved in {attr}`uns['esm2_embeddings'] <anndata.AnnData.uns>`. Thus, we pass the information that `"esm2_embeddings"` stores embeddings of the {attr}`obs['cytokine'] <anndata.AnnData.obs>` treatments via `perturbation_covariate_reps`.

The sample covariate describes the cellular context independent of the perturbation. In our case, these are donors, and given in the {attr}`obs['donor'] <anndata.AnnData.obs>` column. We use the mean of the control sample as donor representation, precomputed and saved in {attr}`uns['donor_embeddings'] <anndata.AnnData.uns>`. We thus pass this piece of information to {class}`~cellflow.model.CellFlow` via `sample_covariate_reps`. 

It remains to define `split_covariates`, according to which {class}`~cellflow.model.CellFlow` trains and predicts perturbations. In effect, `split_covariates` defines how to split the control distributions, and often coincides with `sample_covariates`. This ensure that we don't learn a mapping from the control distribution of donor A to a perturbed population of donor B, but only within the same donor. 

Finally, we can pass `max_combination_length` and `null_value`. These are relevant for combinations of treatments, which doesn't apply for this use case, as we don't want to predict combinationatorial effects of cytokines. In particular, `max_combination_length` is the maximum number of combinations of cytokines which we train on or we want to eventually predict for. The null value is the token representing no treatment, e.g. relevant when we have a treatment with fewer interventions than `max_combination_length`, see tutorials with combinatorial treatments as examples.

In [21]:
cf.prepare_data(
    sample_rep = "X_pca",
    control_key = "is_control",
    perturbation_covariates = {"drug_treatment": ("condition",)},
    perturbation_covariate_reps = {"drug_treatment": 'drug_treatment_onehot'},
    pheno_covariates = ('donor_id_transfer',),
    pheno_covariate_outcomes={"donor_id_transfer": "donor_id_transfer_onehot"},
    max_combination_length = 1,
    null_value = 0.0,
)

[########################################] | 100% Completed | 588.26 ms
[########################################] | 100% Completed | 2.74 ss
[########################################] | 100% Completed | 101.96 ms


In [22]:
layers_before_pool = {
    "drug_treatment": {"layer_type": "mlp", "dims": [1024, 1024], "dropout_rate": 0.5},
    "donor": {"layer_type": "mlp", "dims": [256, 256], "dropout_rate": 0.0},
}

layers_after_pool = {
    "layer_type": "mlp", "dims": [1024, 1024], "dropout_rate": 0.0,
}

We now also explicitly define the `match_fn`:

In [23]:
match_fn = functools.partial(match_linear, epsilon=0.5, tau_a=1.0, tau_b=1.0)

Now we are ready to prepare the model:

In [24]:
cf.prepare_model(
    condition_mode="deterministic",
    regularization=0.0,
    pooling="attention_token",
    pooling_kwargs={},
    layers_before_pool=layers_before_pool,
    layers_after_pool=layers_after_pool,
    condition_embedding_dim=256,
    cond_output_dropout=0.3,
    condition_encoder_kwargs={},
    pool_sample_covariates=True,
    time_freqs=1024,
    time_encoder_dims=[1024, 1024, 1024],
    time_encoder_dropout=0.0,
    hidden_dims=[2048, 2048, 2048],
    hidden_dropout=0.0,
    conditioning="concatenation",
    decoder_dims=[4096, 4096, 4096],
    vf_act_fn=nn.silu,
    vf_kwargs=None,
    probability_path={"constant_noise": 0.5},
    match_fn=match_fn,
    optimizer=optax.MultiSteps(optax.adam(5e-5), 20),
    solver_kwargs={},
    layer_norm_before_concatenation=False,
    linear_projection_before_concatenation=False,
    # Aggregator: defaults are scoring="gated_attn", attn_dim=16, dropout_rate=0.2
    agg_kwargs={"scoring": "gated_attn", "attn_dim": 16},
    # MLP: n_output is auto-set from pheno_data; defaults are n_layers=1, n_hidden=128
    mlp_kwargs={"n_layers": 2, "n_hidden": 128},
)

## Computing and logging metrics during training 

For computing metrics during training, we provide callbacks. We divide callbacks into two categories: The first one performs computations, thus is an instance of {class}`~cellflow.training.ComputationCallback`; the second one are instances of {class}`~cellflow.training.LoggingCallback` and is used for logging. Users can either provide their own callbacks, or make use of existing ones, including {class}`~cellflow.training.Metrics` for computing metrics in the space which the cells are generated in, e.g. in PCA or VAE-space. For computing metrics in gene space, we can use {class}`~cellflow.training.PCADecodedMetrics` in case cells are PCA-embedded, or {class}`~cellflow.training.VAEDecodedMetrics` in case cells are embedding using {class}`~cellflow.external.CFJaxSCVI`. For computing metrics, we can provide user-defined ones, or metrics provided by CellFlow, which we will do below.

For logging, we recommend using [Weights and Biases](https://wandb.ai), for which we provide a callback: {class}`~cellflow.training.WandbLogger`.

As our cells live in PCA-space, we use the {class}`~cellflow.training.PCADecodedMetrics` callback, which takes as input also an {class}`adata <anndata.AnnData>` object which contains the PCs computed from the training data.

In [25]:
metrics_callback = cellflow.training.Metrics(metrics=["r_squared", "mmd", "e_distance"])
decoded_metrics_callback = cellflow.training.PCADecodedMetrics(ref_adata=adata_train, metrics=["r_squared"])
wandb_callback = cellflow.training.WandbLogger(project="cellflow_tutorials", out_dir="~", config={"name": "100m_pbmc"})

# we don't pass the wandb_callback as it requires a user-specific account, but recommend setting it up
callbacks = [metrics_callback, decoded_metrics_callback]


## Training CellFlow

Finally, we are ready to train our model. It just remains to set the number of iterations (this depends on the number of conditions and cells, but should be between 50k and 1m), the batch size (the more heterogeneous the population, the larger), the callbacks which we have defined above, and the frequency of validation steps (note that inference takes relatively long, so once training behaviour is understood for a dataset, we can increase it).

In [ ]:
cf.train(
        num_iterations=500_000,
        batch_size=2, #1024
        callbacks=callbacks,
        valid_freq=700_000,
    )

  0%|          | 181/500000 [1:14:21<3352:01:20, 24.14s/it]

In [ ]:
rng = np.random.default_rng(0)

# Sample one training batch
batch = cf._dataloader.sample(rng)

src   = batch["src_cell_data"]   # (batch_size, n_pca_dims) — control cells
tgt   = batch["tgt_cell_data"]   # (batch_size, n_pca_dims) — true perturbed cells
cond  = batch["condition"]        # {cov_key: array (1, max_combo_len, cov_dim)}

# Predict: transport src cells under the sampled condition
pred = cf.solver.predict_jax(x=src, condition=cond)
# pred: (batch_size, n_pca_dims)

KeyboardInterrupt: 

In [ ]:
from cellflow.data._dataloader import TrainSampler
import jax.numpy as jnp

rng = np.random.default_rng(0)
batch = TrainSampler(data=cf.train_data, batch_size=2).sample(rng)

src  = jnp.array(batch["src_cell_data"])   # must be a JAX array, not numpy
cond = batch["condition"]

# Define the loss as a pure function of src
def loss_fn(src):
    pred = cf.solver.predict_jax(x=src, condition=cond)
    return jnp.mean(pred ** 2)   # replace with your actual loss

# Now grad traces through predict_jax → vmap → ODE solve
grad = jax.grad(loss_fn)(src)


In [ ]:
#pred = cf.solver.predict_jax(x=src, condition=cond)

In [ ]:
len(batch['pheno'])

236

In [ ]:
from cellflow.networks._utils import BaseModule
from typing import Literal
from collections.abc import Callable


class MLP(BaseModule):
    """Fully-connected layers with normalization, dropout, and activation.

    Parameters
    ----------
    n_output
        Number of output features.
    n_layers
        Number of hidden layers.
    n_hidden
        Number of hidden units per hidden layer.
    dropout_rate
        Dropout rate.
    normalization
        Type of normalization. One of ``["layer", "batch", "none"]``.
    act_fn
        Activation function.
    """

    n_output: int
    n_layers: int = 1
    n_hidden: int = 128
    dropout_rate: float = 0.1
    normalization: Literal["layer", "batch", "none"] = "layer"
    act_fn: Callable[[jnp.ndarray], jnp.ndarray] = nn.leaky_relu

    @nn.compact
    def __call__(self, x: jnp.ndarray, training: bool = True) -> jnp.ndarray:
        """Forward computation on ``x``.

        Parameters
        ----------
        x
            Input tensor of shape ``(batch_size, n_input)``.
        training
            Whether the model is in training mode.

        Returns
        -------
        Output tensor of shape ``(batch_size, n_output)``.
        """
        z = x
        for _ in range(self.n_layers):
            z = nn.Dense(self.n_hidden)(z)
            if self.normalization == "layer":
                z = nn.LayerNorm()(z)
            elif self.normalization == "batch":
                z = nn.BatchNorm(use_running_average=not training)(z)
            z = self.act_fn(z)
            z = nn.Dropout(self.dropout_rate)(z, deterministic=not training)
        z = nn.Dense(self.n_output)(z)
        return z


class Aggregator(nn.Module):
    """Aggregator for sets of vectors using various pooling strategies.

    Parameters
    ----------
    scoring
        Pooling method. One of ``["attn", "gated_attn", "mean", "max", "sum"]``.
    attn_dim
        Hidden dimension of the attention layers.
    sample_batch_size
        Bag size used for scaling attention weights when ``scale=True``.
    scale
        Whether to scale attention weights by ``N / sample_batch_size``.
    dropout_rate
        Dropout rate (unused in pooling, reserved for subclasses).
    """

    scoring: Literal["attn", "gated_attn", "mean", "max", "sum"] = "gated_attn"
    attn_dim: int = 16
    sample_batch_size: int | None = None
    scale: bool = False
    dropout_rate: float = 0.2

    @nn.compact
    def __call__(self, x: jnp.ndarray, training: bool = True) -> jnp.ndarray:
        """Forward computation on ``x``.

        Parameters
        ----------
        x
            Input tensor of shape ``(batch_size, N, n_input)``.
        training
            Whether the model is in training mode.

        Returns
        -------
        Pooled tensor of shape ``(batch_size, n_input)``.
        """
        if self.scoring == "sum":
            return jnp.sum(x, axis=-2)
        if self.scoring == "mean":
            return jnp.mean(x, axis=-2)
        if self.scoring == "max":
            return jnp.max(x, axis=-2)

        if self.scoring == "attn":
            # from https://github.com/AMLab-Amsterdam/AttentionDeepMIL/blob/master/model.py
            A = nn.Dense(1, use_bias=False)(jnp.tanh(nn.Dense(self.attn_dim)(x)))  # (batch, N, 1)
        elif self.scoring == "gated_attn":
            # from https://github.com/AMLab-Amsterdam/AttentionDeepMIL/blob/master/model.py
            A_V = jnp.tanh(nn.Dense(self.attn_dim)(x))        # (batch, N, attn_dim)
            A_U = nn.sigmoid(nn.Dense(self.attn_dim)(x))       # (batch, N, attn_dim)
            A = nn.Dense(1, use_bias=False)(A_V * A_U)         # (batch, N, 1)
        else:
            raise NotImplementedError(
                f"scoring={self.scoring!r} is not implemented. "
                'Must be one of ["attn", "gated_attn", "sum", "mean", "max"].'
            )

        A = jnp.swapaxes(A, -1, -2)          # (batch, 1, N)
        A = nn.softmax(A, axis=-1)            # (batch, 1, N)

        if self.scale:
            if self.sample_batch_size is None:
                raise ValueError("sample_batch_size must be set when scale=True.")
            A = A * A.shape[-1] / self.sample_batch_size

        pooled = jnp.matmul(A, x).squeeze(-2)  # (batch, n_input)
        return pooled


In [ ]:
rng = jax.random.PRNGKey(0)
rng_agg, rng_mlp = jax.random.split(rng)

agg = Aggregator(scoring="gated_attn", attn_dim=16)
mlp = MLP(n_output=236, n_layers=2, n_hidden=128)

# --- initialise once ---
dummy_x = pred[None, ...]                                       # (1, batch_size, n_pca)
dummy_emb = agg.init(rng_agg, dummy_x)["params"]               # just to get emb shape
dummy_emb_val = agg.apply({"params": dummy_emb}, dummy_x)      # (1, n_pca)

agg_params = dummy_emb                                          # keep agg params
mlp_params = mlp.init(rng_mlp, dummy_emb_val, training=False)["params"]

# --- reuse from here on, no more init ---
def predict_phenotype(agg_params, mlp_params, pred):
    x = pred[None, ...]
    emb = agg.apply({"params": agg_params}, x)
    return mlp.apply({"params": mlp_params}, emb, training=False)

phenotype_pred = predict_phenotype(agg_params, mlp_params, pred)


In [ ]:
def full_loss(agg_params, mlp_params, src, cond, true_pheno):
    pred      = cf.solver.predict_jax(x=src, condition=cond)   
    emb_aggregated       = agg.apply({"params": agg_params}, pred[None])  
    pheno_hat = mlp.apply({"params": mlp_params}, emb_aggregated,
                          training=False).squeeze(0)            
    return jnp.mean((pheno_hat - true_pheno) ** 2)



loss, (agg_grads, mlp_grads) = jax.value_and_grad(
    full_loss, argnums=(0, 1)
)(agg_params, mlp_params, src, cond, batch['pheno'])